# Managed Colab T4 smoke test
Executed successfully on 2026-10-01 with runtime image 2026.07 and pinned GitHub source a72de2b.
The free GPU pipeline was validated for ten steps. This does not establish agent capability.
Model outputs and complete checkpoint were downloaded and hash-verified locally.
See docs/research/colab-t4-validation-2026-10-01.json for the execution record.


In [ ]:
# Verify the actual free Colab GPU before installing dependencies.
import sys, platform, json
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'A GPU runtime is required'
probe = {'python': sys.version, 'torch': torch.__version__, 'cuda': torch.version.cuda,
         'gpu': torch.cuda.get_device_name(0),
         'vram_gib': torch.cuda.get_device_properties(0).total_memory / 2**30,
         'compute_capability': torch.cuda.get_device_capability(0),
         'bf16': torch.cuda.is_bf16_supported()}
x = torch.ones((16, 16), device='cuda', dtype=torch.float16)
assert torch.isfinite(x @ x).all().item(), 'GPU kernel check failed'
Path('/content/aslm-cloud').mkdir(exist_ok=True)
Path('/content/aslm-cloud/gpu-probe.json').write_text(json.dumps(probe, indent=2))
print(json.dumps(probe, indent=2))
print('GPU kernel check passed. This cell performs no model training.')


In [ ]:
# Pinned source for the free Colab 2026.07 / Python 3.12 / T4 runtime.
import subprocess, hashlib, json, sys
from pathlib import Path
CLOUD = Path('/content/aslm-cloud')
REPO = CLOUD / 'repo'
SOURCE_COMMIT = 'a72de2b4d41a7bf190b5a51d58eb7f46399b6843'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', '--no-checkout', '--branch', 'main', 'https://github.com/Anubhavera/Adaptive-SLM.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'sparse-checkout', 'set', 'training'], check=True)
# Discard only the previously recorded precision patch; it is now committed upstream.
subprocess.run(['git', '-C', str(REPO), 'restore', 'training/finetune_mobile.py'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
ROOT = REPO / 'training'
expected = {
 'finetune_mobile.py': 'fd16ea27494ee38faac3e1a3dc8278ded9e8177d526e3f5355006bf88a93aa43',
 'requirements-mobile.txt': '752684bab3695946e6a576aeb3a7d6c211daf6437269299678f4ea6977b4ec3f',
 'requirements-colab-t4.txt': '65a2bbbbed8dfb8f612a48911e6646a87edfed9b4f1f22b13f9012979aadc849',
 'bundle_checkpoint.py': 'cebb4121b2f813e9743f9377fd9b4e6c9bb7af7f118b8deb0c5b1afe97dedc1f',
 'train.py': '680efe616eada36ba9159e364be7f8ad0755b6bc53673a9ecc70e4f769d18bcf',
 'tests/test_training_regressions.py': '4979a9ead22f6b212a806c499d46ddba4495bf26dfadc7827b557ad20ffae494',
}
for name, sha in expected.items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == sha, 'Source differs: ' + name
(CLOUD / 'source-verification.json').write_text(json.dumps({'commit': SOURCE_COMMIT, 'sha256': expected}, indent=2))
print('Verified pushed main commit:', SOURCE_COMMIT, flush=True)
def run_capture(command, name):
    result = subprocess.run(command, capture_output=True, text=True)
    log = result.stdout + result.stderr
    (CLOUD / (name + '.txt')).write_text(log)
    print(log, flush=True)
    result.check_returncode()
    return result
assert probe['torch'].startswith('2.11.'), 'Recheck TorchAO compatibility for another runtime'
run_capture([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements-colab-t4.txt')], 'dependency-install')
probe['bf16_native'] = torch.cuda.is_bf16_supported(including_emulation=False)
(CLOUD / 'gpu-probe.json').write_text(json.dumps(probe, indent=2))
(CLOUD / 'runtime-extras.json').write_text(json.dumps({'torchao': '0.17.0', 'torch': probe['torch'], 'runtime_image': '2026.07'}, indent=2))
run_capture([sys.executable, '-m', 'pytest', str(ROOT / 'tests'), '-q', '-ra'], 'cloud-regressions')
print('Pinned source, dependencies and all cloud regressions passed.')


In [ ]:
# Inspect pinned data and assistant-only targets before loading model weights.
import os, time, subprocess, json
WORK = CLOUD / 'runs'
RUN = WORK / 'smoke'
COMMON = [sys.executable, str(ROOT / 'finetune_mobile.py'),
 '--model', 'HuggingFaceTB/SmolLM2-360M-Instruct',
 '--model-revision', 'a10cc1512eabd3dde888204e902eca88bddb4951',
 '--dataset', 'HuggingFaceTB/smol-smoltalk',
 '--dataset-revision', 'f73fe857d519ff6ac5af2ea67c4d3834da7b8bcc',
 '--max-samples', '2000', '--max-length', '512', '--batch-size', '1',
 '--grad-accum', '2', '--save-steps', '5', '--max-steps', '10', '--output-dir', str(RUN)]
ENV = os.environ.copy()
ENV['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
ENV['TOKENIZERS_PARALLELISM'] = 'false'
def run_logged(command, name):
    started = time.monotonic()
    with (CLOUD / (name + '.txt')).open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ENV)
        for line in process.stdout:
            log.write(line); log.flush(); print(line, end='', flush=True)
        code = process.wait()
    print(name, 'exit:', code, 'elapsed seconds:', round(time.monotonic() - started, 2), flush=True)
    if code:
        raise subprocess.CalledProcessError(code, command)
run_logged(COMMON + ['--preflight-only'], 'data-preflight')
preflight = json.loads((RUN / 'preflight.json').read_text())
assert preflight['precision'] == 'float16', 'T4 must use native-compatible FP16'
print('Preflight passed. Inspect sample counts before the training cell.')


In [ ]:
# Ten-step GPU smoke test. Snapshots stay ephemeral until downloaded below.
import threading, time
TRANSFER = CLOUD / 'checkpoint-transfer.zip'
stop_snapshot = threading.Event()
def snapshot_latest():
    if not RUN.exists() or not list(RUN.glob('checkpoint-*')):
        return
    result = subprocess.run([sys.executable, str(ROOT / 'bundle_checkpoint.py'), str(RUN), str(TRANSFER)], capture_output=True, text=True)
    with (CLOUD / 'checkpoint-snapshots.txt').open('a') as log:
        log.write(result.stdout + result.stderr)
def snapshot_worker():
    while not stop_snapshot.wait(10):
        snapshot_latest()
worker = threading.Thread(target=snapshot_worker, daemon=True)
worker.start()
try:
    run_logged(COMMON, 'gpu-training')
finally:
    stop_snapshot.set()
    worker.join(timeout=30)
    snapshot_latest()
assert TRANSFER.is_file(), 'No transferable checkpoint was produced'
metrics = json.loads((RUN / 'eval_results.json').read_text())
state = json.loads((RUN / 'checkpoint-10' / 'trainer_state.json').read_text())
assert state['global_step'] == 10
import math
for row in state['log_history']:
    for key in ['loss', 'eval_loss', 'grad_norm']:
        if key in row:
            assert math.isfinite(row[key]), (key, row)
assert math.isfinite(metrics['eval_loss'])
print('GPU smoke PASSED:', json.dumps(metrics, indent=2))
print('Saved complete step-10 checkpoint:', TRANSFER, TRANSFER.stat().st_size, 'bytes')
print('Adapter and merged HF model:', RUN / 'adapter', RUN / 'merged')


In [ ]:
# Package model outputs and evidence for durable local download.
import zipfile, hashlib, importlib.metadata
ARTIFACT = CLOUD / 'adaptive-slm-colab-t4-smoke.zip'
assert TRANSFER.is_file() and (RUN / 'merged' / 'model.safetensors').is_file()
files = [TRANSFER]
files += sorted(p for p in CLOUD.glob('*.json') if p.is_file())
files += sorted(p for p in CLOUD.glob('*.txt') if p.is_file())
files += sorted(p for folder in ['adapter', 'merged'] for p in (RUN / folder).rglob('*') if p.is_file())
files += [RUN / name for name in ['run_manifest.json', 'preflight.json', 'eval_results.json'] if (RUN / name).is_file()]
files += [ROOT / name for name in ['finetune_mobile.py', 'bundle_checkpoint.py', 'requirements-mobile.txt', 'requirements-colab-t4.txt', 'prepare_legacy_converter.py', 'requirements-converter.txt']]
record = {'source_commit': SOURCE_COMMIT, 'run_type': 'ten_step_pipeline_smoke_not_agent_training',
 'gpu': probe, 'metrics': metrics, 'data': preflight['data'], 'checkpoint_step': state['global_step'],
 'packages': {n: importlib.metadata.version(n) for n in ['torch','transformers','datasets','accelerate','peft','torchao']}}
(CLOUD / 'cloud-validation.json').write_text(json.dumps(record, indent=2))
files += [CLOUD / 'cloud-validation.json']
hashes = {}
with zipfile.ZipFile(ARTIFACT, 'w', compression=zipfile.ZIP_STORED) as archive:
    for p in sorted(set(files)):
        name = str(p.relative_to(CLOUD))
        archive.write(p, name)
        hashes[name] = hashlib.sha256(p.read_bytes()).hexdigest()
    archive.writestr('artifact_sha256.json', json.dumps(hashes, indent=2))
with zipfile.ZipFile(ARTIFACT) as archive:
    assert archive.testzip() is None
sha = hashlib.sha256(ARTIFACT.read_bytes()).hexdigest()
(CLOUD / 'adaptive-slm-colab-t4-smoke.zip.sha256').write_text(sha + '  ' + ARTIFACT.name + '\n')
print('Cloud T4 training: PASSED; 10/10 steps; all recorded losses finite.')
print('Checkpoint: complete optimizer/scheduler/RNG/scaler and adapter state.')
print('Export: LoRA adapter and merged HF weights included.')
print('Evidence: pinned source, runtime versions, logs, manifest and checksums included.')
print('Download ready:', ARTIFACT.name, round(ARTIFACT.stat().st_size / 2**20, 2), 'MiB')
print('SHA256:', sha)
print('This validates the training pipeline; agent capability remains to be trained and evaluated.')


In [ ]:
# Smaller durable bundle: full resume state, adapter, source and evidence.
from google.colab import files as colab_files
PORTABLE = CLOUD / 'adaptive-slm-colab-t4-portable.zip'
with zipfile.ZipFile(ARTIFACT) as original, zipfile.ZipFile(PORTABLE, 'w', compression=zipfile.ZIP_STORED) as target:
    kept = {}
    for name in original.namelist():
        if name == 'artifact_sha256.json' or name.startswith('runs/smoke/merged/'):
            continue
        data = original.read(name)
        target.writestr(name, data)
        kept[name] = hashlib.sha256(data).hexdigest()
    target.writestr('artifact_sha256.json', json.dumps(kept, indent=2))
assert zipfile.ZipFile(PORTABLE).testzip() is None
portable_sha = hashlib.sha256(PORTABLE.read_bytes()).hexdigest()
print('Portable training bundle ready:', round(PORTABLE.stat().st_size / 2**20, 2), 'MiB')
print('SHA256:', portable_sha)
print('Contains full step-10 checkpoint, LoRA adapter, reproducibility records and training source.')
colab_files.download(str(PORTABLE))
print('Download requested; save the file locally before ending this runtime.')


In [ ]:
# Verify FP16 optimizer/scaler/RNG restoration on the same T4 runtime.
import shutil
from safetensors.torch import load_file
RESUME_RUN = WORK / 'resume-verification'
assert not RESUME_RUN.exists(), 'Use a fresh verification directory'
RESUME_RUN.mkdir()
shutil.copy2(RUN / 'run_manifest.json', RESUME_RUN / 'run_manifest.json')
shutil.copytree(RUN / 'checkpoint-5', RESUME_RUN / 'checkpoint-5')
resume_command = COMMON[:-1] + [str(RESUME_RUN)]
run_logged(resume_command, 'cuda-resume-verification')
resumed_state = json.loads((RESUME_RUN / 'checkpoint-10' / 'trainer_state.json').read_text())
assert resumed_state['global_step'] == 10
original_tensors = load_file(str(RUN / 'adapter' / 'adapter_model.safetensors'))
resumed_tensors = load_file(str(RESUME_RUN / 'adapter' / 'adapter_model.safetensors'))
assert original_tensors.keys() == resumed_tensors.keys()
max_diff = max((original_tensors[n] - resumed_tensors[n]).abs().max().item() for n in original_tensors)
all_close = all(torch.allclose(original_tensors[n], resumed_tensors[n], rtol=1e-5, atol=1e-6) for n in original_tensors)
resume_record = {'source_commit': SOURCE_COMMIT, 'start_step': 5, 'final_step': resumed_state['global_step'],
 'same_runtime_same_gpu': True, 'different_session_tested': False,
 'exact_tensor_equality': all(torch.equal(original_tensors[n], resumed_tensors[n]) for n in original_tensors),
 'allclose_rtol': 1e-5, 'allclose_atol': 1e-6, 'allclose_passed': all_close,
 'max_absolute_adapter_difference': max_diff,
 'resumed_eval': json.loads((RESUME_RUN / 'eval_results.json').read_text())}
(CLOUD / 'cuda-resume-verification.json').write_text(json.dumps(resume_record, indent=2))
print('RESUME_VERIFICATION_JSON:', json.dumps(resume_record), flush=True)
assert all_close, 'Inspect numerical differences before claiming reproducible resume'
print('CUDA checkpoint resume PASSED.')
